# GoogLeNet — Going Deeper with Convolutions

Implementation of the Inception module concept from Szegedy et al. (2014), applied to MNIST digit classification.

## Dataset

MNIST is used here as a lightweight stand-in for the paper's original ImageNet dataset. It consists of 60,000 training and 10,000 test grayscale images of handwritten digits (0–9) at 28×28 pixels. Images are normalized to zero mean and unit variance using MNIST's dataset-wide statistics.

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim

batch_size = 64
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # normalize(mean, std) for MNIST
])

train_dataset = datasets.MNIST(root='../dataset/mnist/', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
test_dataset = datasets.MNIST(root='../dataset/mnist/', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Inception Module

**Paper:** [Going Deeper with Convolutions](https://arxiv.org/abs/1409.4842) (Szegedy et al., 2014)

The core innovation of GoogLeNet is the **Inception module**: instead of choosing a single filter size, apply multiple filter sizes in parallel and concatenate the results. This lets the network decide which scale of feature is most useful at each layer.

Each Inception module has four parallel branches:

| Branch | Structure | Why |
|---|---|---|
| 1×1 conv | direct | Cheap cross-channel mixing |
| 1×1 → 5×5 conv | dimensionality reduction before large filter | Reduces computation |
| 1×1 → 3×3 → 3×3 conv | two stacked 3×3 ≈ 5×5 receptive field | More non-linearity, fewer params |
| AvgPool → 1×1 conv | pooling branch | Preserves spatial info at each scale |

The 1×1 convolutions before the larger filters act as **bottlenecks** — they reduce the number of channels first, making the 5×5 and 3×3 convolutions much cheaper.

All four branches produce feature maps of the same spatial size, so they can be **concatenated along the channel dimension** (dim=1).

Output channels per branch: 16 + 24 + 24 + 24 = **88 channels**

In [ ]:
class InceptionA(nn.Module):
    def __init__(self, in_channels):
        super(InceptionA, self).__init__()
        # Branch 1: 1x1 conv
        self.branch1x1 = nn.Conv2d(in_channels, 16, kernel_size=1)

        # Branch 2: 1x1 bottleneck -> 5x5 conv
        self.branch5x5_1 = nn.Conv2d(in_channels, 16, kernel_size=1)
        self.branch5x5_2 = nn.Conv2d(16, 24, kernel_size=5, padding=2)

        # Branch 3: 1x1 bottleneck -> 3x3 -> 3x3 (two stacked 3x3 ≈ 5x5 receptive field)
        self.branch3x3_1 = nn.Conv2d(in_channels, 16, kernel_size=1)
        self.branch3x3_2 = nn.Conv2d(16, 24, kernel_size=3, padding=1)
        self.branch3x3_3 = nn.Conv2d(24, 24, kernel_size=3, padding=1)

        # Branch 4: avg pool -> 1x1 conv
        self.branch_pool = nn.Conv2d(in_channels, 24, kernel_size=1)

    def forward(self, x):
        branch1x1 = self.branch1x1(x)

        branch5x5 = self.branch5x5_1(x)
        branch5x5 = self.branch5x5_2(branch5x5)

        branch3x3 = self.branch3x3_1(x)
        branch3x3 = self.branch3x3_2(branch3x3)
        branch3x3 = self.branch3x3_3(branch3x3)

        branch_pool = F.avg_pool2d(x, kernel_size=3, stride=1, padding=1)
        branch_pool = self.branch_pool(branch_pool)

        # concatenate along channel dimension: output = 16+24+24+24 = 88 channels
        return torch.cat([branch1x1, branch5x5, branch3x3, branch_pool], dim=1)

## Network Architecture

A simplified GoogLeNet adapted for MNIST (28×28 grayscale). The original paper used 9 Inception modules on ImageNet; this uses 2 as a concept demonstration.

```
Input (1×28×28)
  → Conv5×5 (1→10) + MaxPool2×2         → (10×12×12)
  → InceptionA(10)                        → (88×12×12)   [16+24+24+24]
  → Conv5×5 (88→20) + MaxPool2×2         → (20×4×4)
  → InceptionA(20)                        → (88×4×4)     [16+24+24+24]
  → Flatten                               → (1408,)
  → Linear(1408→10)                       → class scores
```

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(88, 20, kernel_size=5)  # 88 = output channels of InceptionA

        self.incep1 = InceptionA(in_channels=10)
        self.incep2 = InceptionA(in_channels=20)

        self.mp = nn.MaxPool2d(2)
        self.fc = nn.Linear(1408, 10)  # 88 * 4 * 4 = 1408

    def forward(self, x):
        in_size = x.size(0)
        x = F.relu(self.mp(self.conv1(x)))
        x = self.incep1(x)
        x = F.relu(self.mp(self.conv2(x)))
        x = self.incep2(x)
        x = x.view(in_size, -1)
        x = self.fc(x)
        return x

model = Net().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)

## Training

In [ ]:
def train(epoch):
    running_loss = 0.0
    for batch_idx, data in enumerate(train_loader, 0):
        inputs, target = data
        inputs, target = inputs.to(device), target.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if batch_idx % 300 == 299:
            print('[%d, %5d] loss: %.3f' % (epoch+1, batch_idx+1, running_loss/300))
            running_loss = 0.0


def test():
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print('accuracy on test set: %d %% ' % (100*correct/total))


for epoch in range(10):
    train(epoch)
    test()